In [1]:
from pathlib import Path
import re
import numpy as np

# =============================================================================
# CONFIG
# =============================================================================
ROOT = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/"
    "aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev"
)

COS_BASE = ROOT / "cosines" / "bge_large"
ADS_BASE = ROOT / "llama_summary_1000s"
TITLES_NPZ = ROOT / "embeddings" / "aspectt_vectors.npz"

OUT_DIR = ROOT / "final_like_old_npz_by_month" / "bge_large"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MONTHS = [f"adzuna_month{m:02d}" for m in range(1, 15)]  # edit if you want
OVERWRITE = False

# =============================================================================
# HELPERS
# =============================================================================
def shard_key(p: Path) -> int:
    m = re.search(r"_shard(\d+)$", p.name)
    return int(m.group(1)) if m else 9999

def part_key(p: Path) -> tuple:
    z = np.load(p, allow_pickle=True)
    chunk_start = int(z["chunk_start"]) if "chunk_start" in z.files else -1
    return (chunk_start, p.name)

def concat(arrs, name):
    if not arrs:
        raise RuntimeError(f"No arrays to concat for {name}")
    return np.concatenate(arrs, axis=0)

def load_month_cos(month: str):
    shard_dirs = sorted(COS_BASE.glob(f"{month}_shard*"), key=shard_key)
    if not shard_dirs:
        raise FileNotFoundError(f"No cosine shard dirs found for {month}: {COS_BASE}/{month}_shard*")

    job_ids_all, role_idx_all, role_val_all, task_idx_all, task_val_all = [], [], [], [], []
    rows = 0

    for sd in shard_dirs:
        parts = sorted(sd.glob("part_*.npz"), key=part_key)
        if not parts:
            raise FileNotFoundError(f"No cosine parts in {sd}")

        for part in parts:
            z = np.load(part, allow_pickle=True)
            need = {"job_ids", "role_topk_idx", "role_topk_val", "task_topk_idx", "task_topk_val"}
            missing = need - set(z.files)
            if missing:
                raise KeyError(f"{month} missing keys {missing} in {part}")

            job_ids_all.append(z["job_ids"].astype(str))
            role_idx_all.append(z["role_topk_idx"])
            role_val_all.append(z["role_topk_val"])
            task_idx_all.append(z["task_topk_idx"])
            task_val_all.append(z["task_topk_val"])
            rows += z["job_ids"].shape[0]

        print(f"{month} loaded {sd.name} parts={len(parts)} rows={rows:,}")

    return {
        "job_ids": concat(job_ids_all, "job_ids"),
        "role_topk_idx": concat(role_idx_all, "role_topk_idx"),
        "role_topk_val": concat(role_val_all, "role_topk_val"),
        "task_topk_idx": concat(task_idx_all, "task_topk_idx"),
        "task_topk_val": concat(task_val_all, "task_topk_val"),
    }

# =============================================================================
# LOAD TITLES ONCE
# =============================================================================
assert TITLES_NPZ.exists(), f"Missing titles: {TITLES_NPZ}"
titles = np.load(TITLES_NPZ, allow_pickle=True)["titles"]
n_titles = len(titles)
print("Loaded titles:", n_titles)

# =============================================================================
# BUILD
# =============================================================================
for month in MONTHS:
    out_path = OUT_DIR / f"{month}_final_like_old.npz"
    if out_path.exists() and not OVERWRITE:
        print(f"[SKIP] {month} exists: {out_path.name}")
        continue

    ads_path = ADS_BASE / f"{month}.npz"
    if not ads_path.exists():
        print(f"[MISS] {month} ads not found: {ads_path}")
        continue

    ads = np.load(ads_path, allow_pickle=True)
    need_ads = {"job_ids", "role_text", "taskskill_text", "domains"}
    missing_ads = need_ads - set(ads.files)
    if missing_ads:
        raise KeyError(f"{month} ADS missing keys {missing_ads}: {ads_path}")

    cos = load_month_cos(month)

    # --- alignment checks
    ads_job_ids = ads["job_ids"].astype(str)
    cos_job_ids = cos["job_ids"].astype(str)

    if ads_job_ids.shape[0] != cos_job_ids.shape[0]:
        raise AssertionError(f"{month} row mismatch ads={ads_job_ids.shape[0]:,} cos={cos_job_ids.shape[0]:,}")

    if not np.array_equal(ads_job_ids, cos_job_ids):
        bad = np.nonzero(ads_job_ids != cos_job_ids)[0]
        j = int(bad[0]) if len(bad) else -1
        raise AssertionError(f"{month} job_ids misaligned at row {j}: ads={ads_job_ids[j]} cos={cos_job_ids[j]}")

    role_idx = cos["role_topk_idx"]
    task_idx = cos["task_topk_idx"]

    max_idx = int(max(role_idx.max(), task_idx.max()))
    if max_idx >= n_titles:
        raise AssertionError(f"{month} idx OOB: max_idx={max_idx} titles={n_titles}")

    # --- build candidates per job
    role_val = cos["role_topk_val"]
    task_val = cos["task_topk_val"]

    n = ads_job_ids.shape[0]
    candidates_out = np.empty((n,), dtype=object)

    for i in range(n):
        combo = {}  # title -> list scores

        for idx, val in zip(role_idx[i], role_val[i]):
            t = str(titles[int(idx)])
            combo.setdefault(t, []).append(float(val))

        for idx, val in zip(task_idx[i], task_val[i]):
            t = str(titles[int(idx)])
            combo.setdefault(t, []).append(float(val))

        ranked = sorted(
            ((t, sum(v) / len(v)) for t, v in combo.items()),
            key=lambda x: x[1],
            reverse=True,
        )
        candidates_out[i] = [t for t, _ in ranked]

        if (i + 1) % 200000 == 0:
            print(f"{month} built {i+1:,}/{n:,}")

    np.savez_compressed(
        out_path,
        job_ids=ads_job_ids.astype(object),
        short_desc=ads["role_text"].astype(object),
        tasks_and_skills=ads["taskskill_text"].astype(object),
        domains=ads["domains"].astype(object),
        candidates=candidates_out,
    )

    print(f"[OK] {month} wrote {out_path} rows={n:,}  -> {out_path}")


Loaded titles: 861
[MISS] adzuna_month01 ads not found: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/llama_summary_1000s/adzuna_month01.npz
[MISS] adzuna_month02 ads not found: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/llama_summary_1000s/adzuna_month02.npz
[MISS] adzuna_month03 ads not found: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/llama_summary_1000s/adzuna_month03.npz
[MISS] adzuna_month04 ads not found: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/llama_summary_1000s/adzuna_month04.npz
[MISS] adzuna_month05 ads not found: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/llama_summary_1000s/adzuna_month05.npz
[MISS] adzuna_month06 ads not found: /projects

In [13]:
import numpy as np
from pathlib import Path

p = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/"
    "aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/"
    "final_like_old_npz_by_month/bge_large/"
    "adzuna_month14_final_like_old.npz"
)

assert p.exists(), f"File not found: {p}"

z = np.load(p, allow_pickle=True)

print("keys:", z.files)
print("rows:", len(z["job_ids"]))

# inspect first row
i = 0
print("\njob_id:", z["job_ids"][i])
print("short_desc:", z["short_desc"][i])
print("tasks_and_skills:", z["tasks_and_skills"][i])
print("domain:", z["domains"][i])
print("candidates (top):", z["candidates"][i][:10])


keys: ['job_ids', 'short_desc', 'tasks_and_skills', 'domains', 'candidates']
rows: 1000

job_id: 2919820907
short_desc: [Logistics & Warehouse] We are seeking a Hygiene Operative to join our team.
tasks_and_skills: [Logistics & Warehouse] Maintain a clean and hygienic environment within the warehouse Clean and disinfect all equipment and surfaces Restock and replenish cleaning supplies Report any maintenance issues to the facilities team Ensure compliance with health and safety regulations Attention to detail Physical stamina Teamwork Basic knowledge of cleaning products Ability to follow instructions
domain: Logistics & Warehouse
candidates (top): ['Maintenance Workers, Machinery', 'Medical Equipment Preparers', 'Separating, Filtering, Clarifying, Precipitating, and Still Machine Setters, Operators, and Tenders', 'Facilities Managers', 'Janitors and Cleaners, Except Maids and Housekeeping Cleaners', 'Cleaning, Washing, and Metal Pickling Equipment Operators and Tenders', 'Locker Room,

In [7]:
!rm -rf /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/llm_negative_selection/*


In [10]:
import numpy as np
import json
from pathlib import Path
from datetime import datetime
import time

# =============================================================================
# PATHS
# =============================================================================
DEV_ROOT = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/"
    "aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev"
)

FINAL_DIR = DEV_ROOT / "final_like_old_npz_by_month" / "bge_large"
JSONL_ROOT = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/"
    "aisi_economy_index/store/AISI_demo/stage_1_LLM_summary"
)

OUT_DIR = DEV_ROOT / "llm_negative_selection"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MONTHS = [f"adzuna_month{m:02d}" for m in range(1, 15)]
OVERWRITE = False

print("START:", datetime.now().isoformat(timespec="seconds"), flush=True)
print("FINAL_DIR:", FINAL_DIR, flush=True)
print("JSONL_ROOT:", JSONL_ROOT, flush=True)
print("OUT_DIR:", OUT_DIR, flush=True)

for month in MONTHS:
    print("\n==============================", flush=True)
    print("MONTH:", month, flush=True)

    in_npz = FINAL_DIR / f"{month}_final_like_old.npz"
    jsonl_dir = JSONL_ROOT / f"{month}_npz"
    out_npz = OUT_DIR / f"{month}.npz"

    if out_npz.exists() and not OVERWRITE:
        print("SKIP exists:", out_npz.name, flush=True)
        continue

    if not in_npz.exists():
        print("MISS final_like_old:", in_npz, flush=True)
        continue

    if not jsonl_dir.exists():
        print("MISS jsonl dir:", jsonl_dir, flush=True)
        continue

    # ---- load final npz ----
    with np.load(in_npz, allow_pickle=True) as z:
        required = {"job_ids", "short_desc", "tasks_and_skills", "domains", "candidates"}
        missing = required - set(z.files)
        if missing:
            raise KeyError(f"{month} missing keys {missing}")

        job_ids = z["job_ids"].astype(str)
        job_desc = z["short_desc"].astype(object)
        job_tasks = z["tasks_and_skills"].astype(object)
        domains = z["domains"].astype(object)
        titles = z["candidates"]

    n = len(job_ids)
    job_id_set = set(job_ids)
    print("Rows in final_like_old:", n, flush=True)

    # ---- scan jsonls with progress ----
    title_map = {}
    sector_map = {}
    job_description = {}  # jid -> raw job_description (can be long)

    jsonl_files = sorted(jsonl_dir.glob("*.jsonl"))
    total_files = len(jsonl_files)
    print("JSONL files:", total_files, flush=True)

    found = 0
    t0 = time.time()
    last_print = t0

    for fi, fn in enumerate(jsonl_files, 1):
        with fn.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    obj = json.loads(line)
                except Exception:
                    continue

                jid = obj.get("id")
                if jid is None:
                    continue

                jid = str(jid)
                if jid not in job_id_set:
                    continue

                if jid not in title_map:
                    found += 1

                title_map[jid] = obj.get("title")
                sector_map[jid] = obj.get("category_name")

                # raw full description (if present)
                # NOTE: your json may store it as "description" instead of "job_description"
                job_description[jid] = obj.get("job_description")

        now = time.time()
        if (now - last_print) >= 2.0 or fi == total_files:
            elapsed = now - t0
            rate = fi / elapsed if elapsed > 0 else 0
            eta = (total_files - fi) / rate if rate > 0 else float("inf")
            print(
                f"  [{fi:>3}/{total_files}] {fn.name} | found={found:,}/{n:,} "
                f"| elapsed={elapsed/60:.1f}m ETA={eta/60:.1f}m",
                flush=True
            )
            last_print = now

        if found >= n:
            print("  early-exit: found all ids", flush=True)
            break

    print("Matched JSONL rows:", found, flush=True)

    # ---- align back to npz order ----
    job_ad_title = np.empty(n, dtype=object)
    job_sector_category = np.empty(n, dtype=object)
    job_full_description = np.empty(n, dtype=object)

    miss = 0
    miss_desc = 0

    for i, jid in enumerate(job_ids):
        t = title_map.get(jid)
        c = sector_map.get(jid)
        d = job_full_desc_map.get(jid)

        if t is None and c is None:
            miss += 1
        if d is None:
            miss_desc += 1

        job_ad_title[i] = t
        job_sector_category[i] = c
        job_full_description[i] = d

    np.savez_compressed(
        out_npz,
        job_id=job_ids.astype(object),
        job_desc=job_desc,
        job_tasks=job_tasks,
        domain=domains,
        titles=titles,
        job_ad_title=job_ad_title,
        job_sector_category=job_sector_category,
        job_full_description=job_full_description,
    )

    print(
        f"OK wrote {out_npz.name} rows={n:,} "
        f"missing_title_sector={miss:,} missing_full_desc={miss_desc:,}",
        flush=True
    )

print("\nEND:", datetime.now().isoformat(timespec="seconds"), flush=True)


START: 2026-02-01T01:40:34
FINAL_DIR: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/final_like_old_npz_by_month/bge_large
JSONL_ROOT: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_1_LLM_summary
OUT_DIR: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/llm_negative_selection

MONTH: adzuna_month01
Rows in final_like_old: 1000
JSONL files: 100
  [ 10/100] adzuna_month01_extract_1221286_1249688_20260127_010801.jsonl | found=93/1,000 | elapsed=0.0m ETA=0.3m
  [ 17/100] adzuna_month01_extract_1420100_1448502_20260127_010808.jsonl | found=151/1,000 | elapsed=0.1m ETA=0.4m
  [ 22/100] adzuna_month01_extract_1533708_1562110_20260127_010823.jsonl | found=188/1,000 | elapsed=0.1m ETA=0.4m
  [ 29/100] adzuna_month01_extract_170412_198814_20260127_010758.jsonl | found=248/1,000 | elapsed=0.1m ETA=0.3m
  [ 36/100] adzuna_month01_extract_1

In [11]:
!mv /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/llm_negative_selection/* \
   /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection/


In [12]:
from pathlib import Path
import numpy as np

p = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/"
    "aisi_economy_index/store/AISI_demo/stage_3/dev/"
    "llm_negative_selection/adzuna_month01.npz"
)

assert p.exists(), p
import numpy as np

with np.load(p, allow_pickle=True) as z:
    print("keys:", z.files)
    print("rows:", len(z["job_id"]))

    print("\n--- SAMPLE ROW 0 ---")
    print("job_id:", z["job_id"][0])
    print("job_ad_title:", z["job_ad_title"][0])
    print("job_sector_category:", z["job_sector_category"][0])
    print("domain:", z["domain"][0])
    print("job_desc:", (z["job_desc"][0] or "")[:200])
    print("job_tasks:", (z["job_tasks"][0] or "")[:200])

    print("\n--- CANDIDATES (first 5) ---")
    print(z["titles"][0])



keys: ['job_id', 'job_desc', 'job_tasks', 'domain', 'titles', 'job_ad_title', 'job_sector_category', 'job_full_description']
rows: 1000

--- SAMPLE ROW 0 ---
job_id: 2746781342
job_ad_title: Branch Manager, Shepperton
job_sector_category: Property Jobs
domain: Real Estate
job_desc: [Real Estate] Experienced Branch Manager for a busy estate agency in Shepperton
job_tasks: [Real Estate] Manage the day to day running of the office Motivate team with incentives and rewards Maximise income and profit Increase revenue and profitability List and sell properties Leadership Pr

--- CANDIDATES (first 5) ---
['Agents and Business Managers of Artists, Performers, and Athletes', 'Sales Managers', 'Real Estate Sales Agents', 'Property, Real Estate, and Community Association Managers', 'Real Estate Brokers', 'Securities, Commodities, and Financial Services Sales Agents', 'Advertising Sales Agents']
